# Hierarchical Clustering (Exploratory)

This notebook was used for exploratory work to find a stock universe where the mean-reversion strategy might work.

The idea was to screen several industries, cluster stocks by correlation, and inspect which groups looked homogeneous enough for a cluster mean-reversion strategy.

After trying regional banks, insurance, utilities, and real estate, the clustering and backtests suggested that **REIT—Residential** was the most promising sector. In the end, we simply used the **REIT—Residential** universe for the strategy rather than a hand-picked sub-cluster from the dendrogram.

In [ ]:
import yfinance as yf
from yfinance import EquityQuery
import pandas as pd
import matplotlib.pyplot as plt
import datetime as dt
import numpy as np

industries = [["Financial Services", "Banks—Regional"], ["Financial Services", "Insurance—Property & Casualty"], ["Utilities", "Utilities—Regulated Electric"], ["Real Estate", "REIT—Residential"]]


df_list = []

for industry in industries:
    query = EquityQuery("and", [
        EquityQuery("eq", ["region", "us"]),
        EquityQuery("eq", ["sector", industry[0]]),
        EquityQuery("is-in", ["industry"] + industry[1:]),
        EquityQuery('is-in', ["exchange", "NYQ", "NMS", "NGM", "NCM"]),
        EquityQuery("gte", ["intradaymarketcap", 1_000_000_000]),
        EquityQuery("gte", ["avgdailyvol3m", 500_000]),
    ])

    result = yf.screen(
        query,
        size=250,
        sortField="intradaymarketcap",
        sortAsc=False
    )
    quotes = result["quotes"]
    industry_tickers = [q["symbol"] for q in quotes]

    end_date = dt.datetime.now().strftime('%Y-%m-%d')
    start_date = pd.to_datetime(end_date) - pd.DateOffset(365*10)

    df = yf.download(
    tickers=industry_tickers,
    start=start_date,
    end=end_date
    )['Close']

    print('df size before cleanup: ', df.shape)

    min_days = int(0.95 * len(df))   # allow ~5% missing bars
    valid_tickers = df.columns[df.notna().sum() >= min_days]
    df = df[valid_tickers]

    print('df size after cleanup: ', df.shape)

    df_list.append(df)


# Focus on REIT—Residential for clustering exploration
real_estate_df = df_list[3]

# Ensure tickers are US-listed
for ticker in list(real_estate_df.columns):
    info = yf.Ticker(ticker).get_info()
    if info.get("country") != "United States":
        print('NOT A US STOCK: ', ticker, info.get("country"))
        real_estate_df = real_estate_df.drop(columns=ticker)

df = bank
df.columns


In [ ]:
def plot_stocks(stocks):
    n = len(stocks.columns)
    fig, axes = plt.subplots(n, 1, figsize=(10, 3 * n))

    for ax, ticker in zip(axes, stocks.columns):
        ax.plot(stocks.index, stocks[ticker])
        ax.set_title(ticker)

    plt.tight_layout()
    plt.show()

def scatter_matrix(stocks, figsize=(14, 14)):
    from pandas.plotting import scatter_matrix

    scatter_matrix(
        stocks,
        figsize=figsize,
        alpha=0.4,
        diagonal="hist",
    )
    plt.suptitle("Scatter matrix", y=1.02)
    plt.tight_layout()
    plt.show()

## Hierarchical clustering

Instead of running the strategy on an entire industry at once, we grouped stocks by correlation using hierarchical clustering. To reduce look-ahead bias, clustering was done on one year of data (the year before the backtest window).


In [ ]:
import math

# Selecting the 4th year of data for clustering
end_date = df.index.max() - pd.DateOffset(years=3)
start_date = end_date - pd.DateOffset(years=1)
prices_window = df.loc[start_date:end_date]

# Resample into weekly log returns to lessen noise
weekly_prices = prices_window.resample('W-FRI').last()
weekly_returns = np.log(weekly_prices / weekly_prices.shift()).dropna()

# Correlation matrix
corr = weekly_returns.corr()

# Calculate distance
distance = 1 - corr


# Convert from redundant matrix with repeated stats to an array
condensed_distance = squareform(distance.fillna(0))

# Hierarchical clustering
linkage_matrix = linkage(condensed_distance, method='average')

num_clusters = 1
cut_height = linkage_matrix[len(corr.columns) - num_clusters - 1, 2]
print('Cut height = ', cut_height)

plt.figure(figsize=(16,9))

dendrogram(linkage_matrix, labels=corr.columns.tolist(), color_threshold=cut_height, above_threshold_color='grey')

plt.title('Hierarchical clustering of REIT—Residential stocks')
plt.ylabel('Distance')
plt.show()


# Now that we have a nice hierarichy, let's split it into clusters

cluster_labels = fcluster(
    linkage_matrix,
    t=num_clusters,
    criterion="maxclust"
)

clusters = pd.Series(cluster_labels, index=corr.columns, name="cluster")

# Check which clusters are good
for cluster_id in sorted(clusters.unique()):
    members = clusters[clusters == cluster_id].index.tolist()

    sub_corr = corr.loc[members, members]

    # Average correlation excluding diagonal
    mask = ~np.eye(len(sub_corr), dtype=bool)
    avg_intra_corr = sub_corr.where(mask).stack().mean()

    print(f"Cluster {cluster_id}: {members}")
    print(f"Average intra-cluster correlation: {avg_intra_corr:.2f}")



In [ ]:
import seaborn as sns
cluster_tickers = clusters[clusters == 1].index.tolist()
print(cluster_tickers)
print('Length of cluster tickers: ', len(cluster_tickers))
cluster_weekly_prices = weekly_prices[cluster_tickers]
cluster_weekly_returns = weekly_returns[cluster_tickers]
corr = cluster_weekly_returns.corr() # correlation matrix

cluster_weekly_returns = weekly_returns[cluster_tickers]

scatter_matrix(cluster_weekly_returns)


# Plot larger heatmap
plt.figure(figsize=(14, 12))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    square=True,
)
plt.title("Cluster 83 — weekly return correlation")
plt.tight_layout()
plt.show()


## Conclusion

Although the stocks in this cluster are highly correlated, the mean-reversion strategy did not perform well on them. High correlation ensures the stocks move together, but it does not guarantee that their spread reverts to a mean. A useful improvement would be to base the clustering on metrics more directly tied to mean reversion such as cointegration rather than correlation alone. For the sake of this meta-labelling analysis, I did not take the hierarichal clustering further, so it is an area I could work more on in the future.